In [0]:
%run ../functions/functions

In [0]:
# Nome do database onde a tabela será criada ou utilizada
database_name = "dimensao"

# Nome da tabela destino
table_name = "dm_municipio"

# Caminho alvo no formato database.tabela
target_path = f"{database_name}.{table_name}"

# Nome da chave primária da tabela
pk = "PK_MUNICIPIO"

In [0]:
# Bases utilizadas no relacionamento entre municípios, UF e dados de CNPJ

# Caminho da base de UF e Município utilizada para dados de comércio exterior (comex)
silver_path_uf_mun = f"abfss://silver@stgbbb.dfs.core.windows.net/balancacomercial/UF_MUN/"

# Caminho da base consolidada de municípios extraída do CNPJ
silver_path_municipios_cnpj = f"abfss://silver@stgbbb.dfs.core.windows.net/cnpj/MUNICIPIOS_CONSOLIDADA/"

# Caminho da base consolidada de estabelecimentos extraída do CNPJ
silver_path_estabelecimento = f"abfss://silver@stgbbb.dfs.core.windows.net/cnpj/ESTABELECIMENTOS_CONSOLIDADA/"

In [0]:
# Carrega a base de UF e Municípios do caminho Delta especificado
df_uf_mun= spark.read.format("delta").load(silver_path_uf_mun)

# Carrega a base de municípios consolidados do CNPJ do caminho Delta especificado
df_municipios_cnpj= spark.read.format("delta").load(silver_path_municipios_cnpj)

# Carrega a base de estabelecimentos consolidados do CNPJ do caminho Delta especificado
df_municipios_estabelecimento= spark.read.format("delta").load(silver_path_estabelecimento)

In [0]:
# Cria uma view temporária para a base de municípios e UF do Comex
df_uf_mun.createOrReplaceTempView("df_uf_mun")

# Cria uma view temporária para a base de municípios consolidados do CNPJ
df_municipios_cnpj.createOrReplaceTempView("df_municipios_cnpj")

# Cria uma view temporária para a base de estabelecimentos consolidados do CNPJ
df_municipios_estabelecimento.createOrReplaceTempView("df_municipios_estabelecimento")

In [0]:
query = """
-- CTE para obter o código do município e UF a partir da base de estabelecimentos
WITH mun_uf AS (
  SELECT DISTINCT
    try_cast(municipio as int) as codigo_municipio,
    upper(uf) as uf
  FROM df_municipios_estabelecimento
),

-- CTE para padronizar os nomes dos municípios da base CNPJ para o padrão da base df_uf_mun
cnpj_com_uf AS (
  SELECT
    cn.codigo_municipio,
    -- Aplicando REPLACEs manuais para converter os nomes da base CNPJ para o padrão da base df_uf_mun
    CASE 
      WHEN upper(cn.descricao_municipio) = 'PINGO D''AGUA' THEN 'PINGO-D''AGUA'
      WHEN upper(cn.descricao_municipio) = 'AMPARO DA SERRA' THEN 'AMPARO DO SERRA'
      WHEN upper(cn.descricao_municipio) = 'FLORIANO' THEN 'FLORINIA'
      WHEN upper(cn.descricao_municipio) = 'BARAO DO MONTE ALTO' THEN 'BARAO DE MONTE ALTO'
      WHEN upper(cn.descricao_municipio) = 'EMBU DAS ARTES' THEN 'EMBU'
      WHEN upper(cn.descricao_municipio) = 'MOGI MIRIM' THEN 'MOGI-MIRIM'
      WHEN upper(cn.descricao_municipio) = 'PICARRA' THEN 'PICARRAS'
      WHEN upper(cn.descricao_municipio) = 'IGUARACY' THEN 'IGUARACI'
      WHEN upper(cn.descricao_municipio) = 'ENTRE IJUIS' THEN 'ENTRE-IJUÍS'
      WHEN upper(cn.descricao_municipio) = 'SAO LUIZ DO PARAITINGA' THEN 'SAO LUIS DO PARAITINGA'
      WHEN upper(cn.descricao_municipio) = 'PASSA VINTE' THEN 'PASSA-VINTE'
      WHEN upper(cn.descricao_municipio) = 'SAO VICENTE DO SERIDO' THEN 'SERIDO'
      WHEN upper(cn.descricao_municipio) = 'LAGOA DE ITAENGA' THEN 'LAGOA DO ITAENGA'
      WHEN upper(cn.descricao_municipio) = 'GRACCHO CARDOSO' THEN 'GRACHO CARDOSO'
      WHEN upper(cn.descricao_municipio) = 'PINDARE MIRIM' THEN 'PINDARE-MIRIM'
      WHEN upper(cn.descricao_municipio) = 'SAO FRANCISCO DE ASSIS DO PIAUI' THEN 'SAO FRANCISCO DE ASSIS PIAUI'
      WHEN upper(cn.descricao_municipio) = 'POXOREU' THEN 'POXOREO'
      WHEN upper(cn.descricao_municipio) = 'BELEM DO SAO FRANCISCO' THEN 'BELEM DE SAO FRANCISCO'
      WHEN upper(cn.descricao_municipio) = 'ITAPAJE' THEN 'ITAPAGE'
      WHEN upper(cn.descricao_municipio) = 'OLHO D''AGUA DO BORGES' THEN 'OLHO-D''AGUA DO BORGES'
      WHEN upper(cn.descricao_municipio) = 'SAO TOME DAS LETRAS' THEN 'SAO THOME DAS LETRAS'
      ELSE upper(cn.descricao_municipio)
    END as municipio,
    mu.uf
  FROM df_municipios_cnpj cn
  LEFT JOIN mun_uf mu
    ON cast(cn.codigo_municipio as int) = cast(mu.codigo_municipio as int)
)

-- Seleção final dos municípios, unindo as bases padronizadas
SELECT DISTINCT
  m.CO_MUN_GEO,      -- Código geográfico do município na base df_uf_mun
  m.NO_MUN,          -- Nome do município na base df_uf_mun
  m.SG_UF,           -- Sigla da UF na base df_uf_mun
  cn.codigo_municipio, -- Código do município na base CNPJ
  cn.municipio,        -- Nome do município padronizado da base CNPJ
  cn.uf                -- Sigla da UF da base CNPJ
FROM df_uf_mun m
LEFT JOIN cnpj_com_uf cn
  ON upper(m.NO_MUN) = upper(cn.municipio)
 AND upper(m.SG_UF) = upper(cn.uf)
"""

In [0]:
# Executa a query SQL definida anteriormente e armazena o resultado no DataFrame df_final
df_final = spark.sql(query)

In [0]:
# Cria o banco de dados se ele não existir
spark.sql(f"CREATE DATABASE IF NOT EXISTS database_name")

In [0]:
# Salva o DataFrame df_final como uma tabela Hive no caminho especificado por target_path,
# utilizando a coluna pk como chave primária.
save_hive_table(df_final, target_path, pk)